In [39]:
from pathlib import Path
import mne
import numpy as np
import json
import csv
import tensorflow as tf
import sys

PROJECT_ROOT = Path.cwd().resolve()

# Nach oben gehen, bis das Datenverzeichnis gefunden wird
while not (PROJECT_ROOT / "data" / "raw" / "ds005363-download").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Projektverzeichnis nicht gefunden.")

    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "ds005363-download"

BUNDLE_DIR = PROJECT_ROOT / "notebooks" / "BunDLe-Net"
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))

print("Arbeitsverzeichnis:", Path.cwd())
print("Datenverzeichnis:", DATA_ROOT)

def read_tsv(path: Path) -> list[dict[str, str]]:
    """Liest eine TSV-Datei robust gegen ein UTF-8-BOM ein."""

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file, delimiter="\t"))

def first_matching(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Keine Datei mit Muster {pattern!r} in {directory}")
    return matches[0]

def subject_directory(data_root: Path, subject: str) -> Path:
    subject_label = subject if subject.startswith("sub-") else f"sub-{subject}"
    directory = data_root / subject_label / "ses-01" / "eeg"
    if not directory.is_dir():
        raise FileNotFoundError(f"EEG-Verzeichnis nicht gefunden: {directory}")
    return directory

def read_data(subject: str, duration: float, data_root: Path, output_root: Path): 
    eeg_dir = subject_directory(data_root, subject)

    vhdr_path = first_matching(eeg_dir, "*_eeg.vhdr")
    json_path = first_matching(eeg_dir, "*_eeg.json")
    channels_path = first_matching(eeg_dir, "*_channels.tsv")
    events_path = first_matching(eeg_dir, "*_events.tsv")

    with json_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)
    channels = read_tsv(channels_path)
    events = read_tsv(events_path)

    print(f"Lese BrainVision-Aufnahme ein: {vhdr_path}")
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    if len(eeg_picks) == 0:
        raise RuntimeError("Die Aufnahme enthält keine als EEG markierten Kanäle.")

    return raw, channels, events



Arbeitsverzeichnis: /Users/benediktadler/Documents/Uni Projekte/bachelor-thesis/notebooks/BunDLe-Net/notebooks
Datenverzeichnis: /Users/benediktadler/Documents/Uni Projekte/bachelor-thesis/data/raw/ds005363-download


In [40]:
try:
    raw, channels, events = read_data(
        subject="101",
        duration=60.0,
        data_root=DATA_ROOT,
        output_root=PROJECT_ROOT / "results",
    )

except FileNotFoundError as exc:
    raise SystemExit(f"Datei nicht verfügbar: {exc}") from exc
except RuntimeError as exc:
    raise SystemExit(f"Analyse konnte nicht durchgeführt werden: {exc}") from exc


stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5

def extract_s(trial_type):
    if not trial_type.startswith("Stimulus/S"):
        return None

    number = trial_type.removeprefix("Stimulus/S").strip()
    return int(number)

labels = []

for event in events:
    code = extract_s(event["trial_type"])

    if code in stimulus_to_state:
        label = stimulus_to_state[code]
    elif code == 199 or code in range(101, 109):
        label = 0  
    else:
        continue 

    sample = int(event["sample"])
        
    labels.append((sample, label))

import queue
import threading

def prep_data(X, B, win=15):
    if win < 1 or len(X) <= win:
        raise ValueError("X must contain more samples than the positive window length.")
    if len(X) != len(B):
        raise ValueError("X and B must have the same number of samples.")
    # Read-only views share the original recording, including overlapping pairs.
    windows = np.lib.stride_tricks.sliding_window_view(
        X, window_shape=win + 1, axis=0
    ).transpose(0, 2, 1)
    X_paired = np.lib.stride_tricks.sliding_window_view(
        windows, window_shape=win, axis=1
    ).transpose(0, 1, 3, 2)
    B_1 = B[win:]
    print(f'Prepped data, X shape {X_paired.shape}, B shape {B_1.shape}')
    return X_paired, B_1



def generate_B(start, neuron_traces_block):
    B = np.zeros(neuron_traces_block.shape[1], dtype=int)
    l = len(labels)

    for e in range(l):
        current_sample = max(0, labels[e][0] - start) 
        label = labels[e][1]

        if e + 1 < l:
            next_sample = min(len(B), labels[e + 1][0] - start)
        else:
            next_sample = len(B)

        if current_sample < next_sample:
            B[current_sample:next_sample] = label

    return B




Lese BrainVision-Aufnahme ein: /Users/benediktadler/Documents/Uni Projekte/bachelor-thesis/data/raw/ds005363-download/sub-101/ses-01/eeg/sub-101_ses-01_task-objrecog_eeg.vhdr


In [ ]:
from functions import _pca_initialisation, tf_batch_prep, _best_of_5_runs, BunDLeTrainer, tqdm, BunDLeNet
import matplotlib.pyplot as plt

batch_size = 1024
win = 150
block_size = 16*batch_size

model = BunDLeNet(latent_dim=3)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
trainer = BunDLeTrainer(model, optimizer)
gamma=0.09 
all_losses = []


def train_block(X_paired, B_1, batch_size=batch_size):
    perm = np.random.permutation(len(X_paired))   # Shuffling the data using a random permutation
    for s in range(0, len(perm), batch_size):

        idx = perm[s:s + batch_size]

        x = tf.convert_to_tensor(X_paired[idx], dtype=tf.float32)
        b = tf.convert_to_tensor(B_1[idx], dtype=tf.int64)

        DCC_loss, behaviour_loss, total_loss = trainer.train_step(x, b, gamma=gamma)

        all_losses.append([float(DCC_loss), float(behaviour_loss), float(total_loss)])

def consume_data():
    while True:
        item = q.get()
        try:
            if item is None:
                return
            start, block = item
            B = generate_B(start, block)
            X_paired, B_1 = prep_data(block.T, B, win=win)
            train_block(X_paired, B_1)
            print(f"Block ab {start} fertig, letzter Loss: {all_losses[-1]}")
        finally:
            q.task_done()

q = queue.Queue(maxsize=2)

def fetch_data(): 
    for start in range(0, raw.n_times, block_size):
        stop = min(start+block_size, raw.n_times)
        neuron_traces_block = raw.get_data(picks = "eeg", start=start, stop=stop)
        print(f'Putting item into queue')
        q.put((start, neuron_traces_block))
    q.put(None)

for epoch in range(0, 1):
    producer = threading.Thread(target=fetch_data, daemon=True)
    consumer = threading.Thread(target=consume_data, daemon=True)
    producer.start(); consumer.start()
    producer.join(); consumer.join()
    print("Finished)")

loss_array = np.array(all_losses)

plt.figure()
for i, label in enumerate(
    [
        "$\mathcal{L}_{{Markov}}$",
        "$\mathcal{L}_{{Behavior}}$",
        "Total loss $\mathcal{L}$",
    ]
):
    plt.semilogy(loss_array[:, i], label=label)
plt.legend()
plt.show()

Putting item into queue
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Putting item into queue
Putting item into queue
Block ab 0 fertig, letzter Loss: [0.001260910532437265, 1.8342680931091309, 1.8355289697647095]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 16384 fertig, letzter Loss: [0.0029016437474638224, 1.8648887872695923, 1.8677904605865479]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 32768 fertig, letzter Loss: [0.0030119267757982016, 1.873819351196289, 1.876831293106079]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 49152 fertig, letzter Loss: [0.00381826376542449, 1.8322771787643433, 1.8360954523086548]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 65536 fertig, letzter Loss: [0.0012466938933357596, 1.8699674606323242, 1.8712141513824463]
Prepped data

KeyboardInterrupt: 

Block ab 2949120 fertig, letzter Loss: [0.0008627030765637755, 2.0649521350860596, 2.065814733505249]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 2965504 fertig, letzter Loss: [0.0009028944768942893, 1.9759531021118164, 1.976855993270874]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 2981888 fertig, letzter Loss: [0.0009051479864865541, 1.8624886274337769, 1.863393783569336]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 2998272 fertig, letzter Loss: [0.0008117167744785547, 1.8128435611724854, 1.8136552572250366]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 3014656 fertig, letzter Loss: [0.0006862090085633099, 1.7758898735046387, 1.776576042175293]
Prepped data, X shape (16234, 2, 150, 64), B shape (16234,)
Putting item into queue
Block ab 3031040 fertig, letzter Loss: [0.0006670221337117255, 1.6960

In [ ]:


# EEG-Kanäle auswählen
eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])

# Messwerte: (Kanäle, Zeitpunkte)
neuron_traces = raw.get_data(picks=eeg_picks)

# Kanalnamen in derselben Reihenfolge wie die Messwerte
channel_names = np.array(
    [raw.ch_names[i] for i in eeg_picks],
    dtype=object
)

# Abtastrate der Aufnahme: bei dir 1000 Hz
fps = raw.info["sfreq"]
state_names = [
    "keine Kategorie",
    "Gesicht",
    "Tier",
    "Ort",
    "Objekt",
    "Catch",
]

stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5
print("--------------------------------------- STIMULUS TO STATE ARRAY ---------------------------------------")
print(stimulus_to_state)
print("")
print("")

event = events[4] 

sample = int(event["sample"])
messwerte = neuron_traces[:, sample]

print("--------------------------------------- Neuron Traces ---------------------------------------")
print(neuron_traces[:,:])
print("")
print(neuron_traces[[0],:])
print("")
print(neuron_traces.shape)

print("--------------------------------------- EVENT NR. 4 ---------------------------------------")
print(event)
print("")
print("")
print("--------------------------------------- MESSWERTE FÜR SAMPLE VON EVENT NR. 4 ---------------------------------------")
print(messwerte)
print("")
print("")
print("--------------------------------------- FORM VON MESSWERTE ---------------------------------------")
print(messwerte.shape)
print("")
print("")
print("--------------------------------------- FORM VON NEURON TRACES ---------------------------------------")
print(neuron_traces.shape)
print("")
print("")
# print("--------------------------------------- FORM VON B ---------------------------------------")
# B = np.zeros(neuron_traces.shape[1], dtype=int)
# print(B.shape)